# Embedding and clustering of a model ensemble — fixed-kernel encoder, UMAP, DBSCAN, cluster viewer

This notebook runs, on one ensemble, the three steps that were spread over `2b_Encoding_with_3D_Autoencoder_topology_red.ipynb`, `test_notebook3.ipynb` and `test_notebook2_original_file.ipynb`:

1. **features** — each model $m_i$ ($i = 1, \dots, N$), a grid of $n_z \times n_y \times n_x$ density values, is mapped to a vector $\mathbf{f}_i \in \mathbb{R}^{523}$ made of 512 values from the fixed-kernel encoder of the 2b notebook, 6 topological scalars (`topo_features.py`) and 5 moments of the density distribution; the columns are standardised;
2. **embedding and clustering** — the vectors are embedded in 2D with UMAP (McInnes et al., 2018) and the embedding is clustered with DBSCAN (Ester et al., 1996), as in `test_notebook3`;
3. **cluster statistics and viewer** — for each cluster $c$, the median model $\tilde m_c(\mathbf{x}) = \mathrm{median}_{i \in c}\, m_i(\mathbf{x})$ and the standard-deviation model $s_c(\mathbf{x}) = \big(\frac{1}{|c|}\sum_{i \in c} (m_i(\mathbf{x}) - \bar m_c(\mathbf{x}))^2\big)^{1/2}$, where $\bar m_c$ is the mean model and $|c|$ the number of members, are computed and shown in tile views and in a self-contained interactive page (Bokeh), in the spirit of the linked-brushing mosaic of `test_notebook2`.

### Data

With `USE_SYNTHETIC = True`, the ensemble is the synthetic stand-in of the motif workflow (`synthetic_ensemble.py`, as in `motif_workflow_boulia_v2.ipynb`): three chains of undercover models (cover / sediments / basement, with sampled unit densities) with born lenses, dykes and pipes and basement highs of random geometry, and a truth table recording what was drawn in each model. The truth table is used at the end to read the clusters. With `USE_SYNTHETIC = False`, the Boulia file is loaded as in the other notebooks.

Shapes: `all_models` is $(n_z, n_y, n_x, N)$ with depth along axis 0 (index 0 at the top) and the model index last; `core` is the analysis region (lateral padding removed), on which all features are computed. Sections are named by the axis held fixed: the section at $y = i_y$ is an $x$–$z$ panel, the section at $x = i_x$ a $y$–$z$ panel, the map at $z = i_z$ an $x$–$y$ panel.

### The encoder

The encoder of the 2b notebook has no trained weight: it applies 351 fixed $3 \times 3 \times 3$ difference kernels (one per unordered pair of voxels of the neighbourhood, $+1/\sqrt{2}$ at one voxel and $-1/\sqrt{2}$ at the other), takes the absolute value and max-pools, three times (the second and third blocks depthwise), average-pools the output of each block, projects each pooled output on a fixed random column-orthonormal basis (128, 128 and 256 dimensions; Johnson & Lindenstrauss, 1984) and $L_2$-normalises each branch. The response of a difference kernel is non-zero only across unit contacts, so the features measure the position and density of the interfaces at three scales. `ensemble_features.py` computes it with NumPy by shifting the grid instead of convolving it, with the same flatten order and the same random seeds as the TensorFlow model, so that no GPU and no TensorFlow are needed; `ef.check_against_convolution()` compares it with a brute-force convolution (agreement to $10^{-7}$, float32 rounding).

All parameters are set in the cells where they are used and named in capitals. Cells are meant to be run from top to bottom. Modules: `synthetic_ensemble.py` (data), `ensemble_features.py` (features), `cluster_tools.py` (embedding, clustering, statistics, figures), `cluster_viewer.py` (interactive page); `topo_features.py`, `pca_analysis.py` and `t2_analysis.py` are the user's own modules.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import time
from importlib import reload

import ensemble_features as ef
import cluster_tools as ct
import cluster_viewer as cv
for m_ in (ef, ct, cv):
    reload(m_)

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
plt.rcParams['figure.dpi'] = 110
print("Imports: completed.")

## Data

In [ ]:
USE_SYNTHETIC = True
MAX_NUM_MODELS = None        # cap on the number of models (None = all), evenly spread over the ensemble
FILE_PREFIX = 'synthetic_3chains'   # prefix of the files written by this notebook (features, embedding, viewer)

if USE_SYNTHETIC:
    import synthetic_ensemble as se
    N_PER_CHAIN = 1000        # 3 chains -> 1200 models
    padd = 6
    all_models, chain_id, data_misfit, truth = se.make_ensemble(n_per_chain=N_PER_CHAIN, padd=padd, seed=0,
                                                                return_truth=True)
else:
    # --- Boulia: data_models_5chains.npy as in test_notebook3 / motif_workflow_boulia_v2 ---
    pth_rt = r"C:\Users\00090846\OneDrive - UWA\Documents\case_studies\transd\boulia\transd_chains"
    chain_names = ['20240411_field_re_5869', '20240416_field_re_4192', '20240428_field_re_4493',
                   '20240903_field_re_758', '20241106_field_re_9108']
    FILE_AXES = (2, 1, 0)    # positions in the file of (depth, y, x): the Boulia file is (x, y, depth, N)
    padd = 12
    truth = None
    # (ensemble_io.load_ensemble of the motif workflow does the same with a model-major copy, which is
    #  faster and avoids the memmap problem of OneDrive-synced folders)
    a = np.load(os.path.join(pth_rt, 'data_models_5chains.npy'), mmap_mode='r')
    N_file = a.shape[3]
    keep = np.arange(N_file) if MAX_NUM_MODELS is None or MAX_NUM_MODELS >= N_file else \
        np.unique(np.linspace(0, N_file - 1, MAX_NUM_MODELS).round().astype(int))
    chain_id = np.repeat(np.array(chain_names), N_file // len(chain_names))[keep]
    all_models = np.ascontiguousarray(np.transpose(np.asarray(a[..., keep], dtype=np.float32), tuple(FILE_AXES) + (3,)))
    data_misfit = np.full(len(keep), np.nan)
    FILE_PREFIX = 'boulia_5chains'

if MAX_NUM_MODELS is not None and MAX_NUM_MODELS < all_models.shape[3]:
    keep = np.unique(np.linspace(0, all_models.shape[3] - 1, MAX_NUM_MODELS).round().astype(int))
    all_models, chain_id, data_misfit = all_models[..., keep], chain_id[keep], data_misfit[keep]
    if truth is not None:
        truth = truth.iloc[keep].reset_index(drop=True)

core = np.ascontiguousarray(all_models[:, padd:all_models.shape[1] - padd, padd:all_models.shape[2] - padd, :])
nz, ny, nx, N = core.shape
print("all_models:", all_models.shape, "-> core (padding removed):", core.shape)
print("chains:", dict(zip(*np.unique(chain_id, return_counts=True))))
if truth is not None:
    print("bodies drawn (fraction of models):", truth[['high', 'lens', 'dyke', 'pipe']].mean().round(2).to_dict())

## Features

The three groups of features are computed on `core`: the encoder on the densities scaled to $[0, 1]$ with the global minimum and maximum of the ensemble (as in the 2b notebook), the topological scalars and the density moments on the raw values. The result is standardised column by column (`StandardScaler`), so that the 512 encoder values, which are small and correlated, do not swamp the 11 others, and saved with the same file names as in the 2b notebook. On a laptop, the encoder takes about 0.2 s per model of $36 \times 42 \times 52$ cells (the random projections are built once, a few seconds).

In [ ]:
t0 = time()
enc = ef.DiffNetEncoder(core.shape[:3])                 # fixed random projections for this grid (seeds 1, 2, 3)
print(f"encoder: pooled dimensions {enc.in_dims} -> {enc.dims} ({time() - t0:.1f} s to build the projections)")

t0 = time()
latent_codes = enc.encode_ensemble(core, n_jobs=-1)     # (N, 512)
print(f"latent codes: {latent_codes.shape} in {time() - t0:.0f} s")

t0 = time()
topo, features_density = ef.topo_and_density_features(core, n_jobs=-1)    # (N, 6), (N, 5)
print(f"topological and density features: {topo.shape}, {features_density.shape} in {time() - t0:.0f} s")

features, scaler = ef.combine_features(latent_codes, topo, features_density)   # (N, 523), standardised
print("combined features:", features.shape)

np.save(f"{FILE_PREFIX}_features_latent_codes.npy", latent_codes)
np.save(f"{FILE_PREFIX}_topo.npy", topo)
np.save(f"{FILE_PREFIX}_features_density.npy", features_density)
np.save(f"{FILE_PREFIX}_features.npy", features)
pd.DataFrame(np.hstack([topo, features_density]), columns=ef.TOPO_FEATURE_NAMES + ef.DENSITY_FEATURE_NAMES).describe().round(3)

### Which features carry the variance

As in `test_notebook3`, a PCA of the standardised features ranks the original columns by their weighted loading on the principal components (`pca_analysis.get_top_features`); the table counts, among the 50 most important columns, how many belong to each group. Hotelling's $T^2$ (`t2_analysis`) flags the models farthest from the centre of the feature cloud: the `T2_OUTLIER_FRACTION` most extreme models are listed, as candidates for exclusion before the embedding (the notebook keeps them).

In [ ]:
# import pca_analysis as pca
# import t2_analysis as t2
# from collections import Counter

# pca_obj, _ = pca.compute_pca(features, scale=False)      # already standardised
# cum = np.cumsum(pca_obj.explained_variance_ratio_)
# print(f"components for 50 / 90 % of the variance: {np.searchsorted(cum, 0.5) + 1} / {np.searchsorted(cum, 0.9) + 1}")
# idx, vals = pca.get_top_features(pca_obj, k=50)
# print("groups of the 50 most important columns:", dict(Counter(ef.classify_feature(i) for i in idx)))

# T2_OUTLIER_FRACTION = 0.01
# res = t2.analyze_t2(features, outlier_fraction=T2_OUTLIER_FRACTION, scale=False, plot=False)
# print(f"T2 outliers ({T2_OUTLIER_FRACTION:.0%} most extreme models): {res['outliers']}")

## UMAP embedding

The parameters are those of the 2b notebook: `n_neighbors = 5` (local structure; the embedding then forms thin strings of neighbouring models rather than blobs), `min_dist = 0.01` (tight packing), cosine metric on the standardised features, fixed seed. The embedding is saved as a CSV with the columns `comp_1`, `comp_2` followed by the features, as the 2b notebook writes it, so that `test_notebook3` can read it.

In [ ]:
UMAP_N_NEIGHBORS = 3
UMAP_MIN_DIST = 0.05
UMAP_METRIC = 'cosine'

t0 = time()
coords, reducer = ct.umap_embed(features, n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST, metric=UMAP_METRIC,
                                random_state=42)
print(f"UMAP: {coords.shape} in {time() - t0:.0f} s")

df_umap = pd.concat([pd.DataFrame(coords, columns=['comp_1', 'comp_2']), pd.DataFrame(features)], axis=1)
umap_csv = f"{FILE_PREFIX}_umap_{UMAP_N_NEIGHBORS}Features_cnn_topo_dens_DiffNN_{UMAP_MIN_DIST}_metrics_{UMAP_METRIC}__.csv"
df_umap.to_csv(umap_csv, index=False)
print("saved:", umap_csv)

fig, ax = plt.subplots(figsize=(7, 6))
for c in np.unique(chain_id):
    m = chain_id == c
    ax.scatter(coords[m, 0], coords[m, 1], s=6, alpha=0.7, label=c)
ax.set_aspect('equal'); ax.legend(markerscale=3, fontsize=8); ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title('UMAP embedding, coloured by chain')
plt.show()

## Clustering

DBSCAN groups the points of the embedding that are chained by neighbours closer than `EPS`, with at least `MIN_SAMPLES` neighbours for a point to be a core point; points reachable from no core point are noise (label $-1$). `EPS` is in embedding units and depends on $N$ and on the UMAP parameters: the $k$-distance curve (distance of each point to its `MIN_SAMPLES`-th neighbour, sorted) gives the usual choice, at the knee of the curve, and the cell prints the number of clusters and of noise points for a few values before using `EPS`. Setting `CLUSTERER = 'hdbscan'` uses HDBSCAN instead (Campello et al., 2013), which needs no `EPS` but a minimum cluster size.

In [ ]:
CLUSTERER = 'dbscan'        # 'dbscan' or 'hdbscan'
EPS = 0.35                  # DBSCAN: maximum distance between neighbours, in embedding units
MIN_SAMPLES = 5             # DBSCAN: neighbours for a core point; HDBSCAN: min_samples
MIN_CLUSTER_SIZE = 10       # HDBSCAN only

for eps in (0.2, 0.3, 0.35, 0.5, 0.75):
    n_cl, n_noise = ct.cluster_summary(ct.cluster_dbscan(coords, eps=eps, min_samples=MIN_SAMPLES))
    print(f"eps = {eps:<5} -> {n_cl:3d} clusters, {n_noise:4d} noise points")
ct.plot_kdist(coords, k=MIN_SAMPLES, eps=EPS)
plt.show()

if CLUSTERER == 'dbscan':
    labels = ct.cluster_dbscan(coords, eps=EPS, min_samples=MIN_SAMPLES)
else:
    labels = ct.cluster_hdbscan(coords, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES)
n_clusters, n_noise = ct.cluster_summary(labels)
print(f"{CLUSTERER}: {n_clusters} clusters, {n_noise} noise points, N = {N}")
np.save(f"{FILE_PREFIX}_cluster_labels.npy", labels)

ct.plot_embedding(coords, labels, min_size=MIN_SAMPLES)
plt.show()

### Cluster table

One row per cluster, sorted by size: number of models, share of each chain, mean voxel standard deviation (the spread of the cluster in model space, in the density units of the models: small values mean that the members are nearly the same model), index of the representative model (the member closest to the cluster centroid in the embedding), and, for the synthetic ensemble, the fraction of members carrying each body. The enrichment $P(\text{body} \mid c) / P(\text{body})$ compares the frequency of a body in cluster $c$ with its frequency in the whole ensemble, as done with motifs in the motif workflow: a value well above 1 identifies what the cluster is made of, a value near 0 what it excludes.

In [ ]:
tab = ct.cluster_table(core, coords, labels, chain_id, truth=truth, min_size=MIN_SAMPLES)
display(tab.round(2).head(20))
if truth is not None:
    enr = ct.truth_enrichment(truth, labels, min_size=MIN_SAMPLES)
    display(enr.round(2).head(20))

## Cluster statistics: tile views and sections

The tile views show one section of the median model of every cluster (clusters sorted by size, as in `test_notebook3`), and the same for the standard-deviation model. `IY`, `IX`, `IZ` are the indices of the sections used here and in the viewer; for the synthetic ensemble, the basement high is centred near $y \approx 18$, $x \approx 16$ of `core` and the pipe near $y \approx 26$, $x \approx 39$, so `IY = 22` passes between them and the depth map at `IZ = 10` cuts the lenses, dykes, pipes and most basement highs.

In [ ]:
IY, IX, IZ = 22, nx // 2, 10          # section indices in core (y = IY: x-z panel; x = IX: y-z panel; z = IZ: x-y map)
VMIN, VMAX = np.percentile(core[..., ::max(N // 200, 1)], [0.5, 99.5])   # common colour scale for densities

fig = ct.plot_cluster_tiles(core, labels, orient='y', iy=IY, stat='median', min_size=MIN_SAMPLES, max_tiles=30,
                            vmin=VMIN, vmax=VMAX)
fig.savefig(f"{FILE_PREFIX}_cluster_tiles_median_y{IY}.png", dpi=200, bbox_inches='tight')
plt.show()
fig = ct.plot_cluster_tiles(core, labels, orient='y', iy=IY, stat='std', min_size=MIN_SAMPLES, max_tiles=30,
                            vmin=0, cmap='magma')
plt.show()

In [ ]:
# Median, standard deviation and representative model of the largest clusters, on the y section and on the depth map.
N_SHOW = 4
for cl in ct.cluster_ids_by_size(labels, MIN_SAMPLES)[:N_SHOW]:
    for orient in ('y', 'z'):
        fig = ct.plot_cluster_sections(core, coords, labels, cl, orient=orient, iy=IY, ix=IX, iz=IZ,
                                       vmin=VMIN, vmax=VMAX)
        plt.show()

## Interactive viewer

`cluster_viewer.build_viewer` writes a self-contained HTML page: the embedding coloured by cluster (hover: the $y$ section of the model under the cursor, its cluster, its chain and, for the synthetic ensemble, the bodies it contains; tap: select the cluster), a drop-down list of the clusters sorted by size, a choice of section ($y$, $x$ or depth map), the median, standard-deviation and representative models of the selected cluster on that section, and the cluster table. Clusters with fewer than `MIN_SAMPLES` models are shown as grey points but not listed. The saved page is embedded below the cell (open the HTML file in a browser for a full-size view).

In [ ]:
from IPython.display import IFrame

viewer_html = f"{FILE_PREFIX}_cluster_viewer.html"
layout = cv.build_viewer(core, coords, labels, chain_id, iy=IY, ix=IX, iz=IZ, truth=truth,
                         min_size=MIN_SAMPLES, max_clusters=60, vmin=VMIN, vmax=VMAX,
                         title=f'{FILE_PREFIX}: {n_clusters} clusters ({CLUSTERER})')
cv.save_viewer(layout, viewer_html, title=f'{FILE_PREFIX} cluster viewer')
print("saved:", viewer_html, f"({os.path.getsize(viewer_html) / 1e6:.1f} MB)")
IFrame(viewer_html, width=1250, height=1150)      # the saved page, embedded; open the file for a full-size view

## Notes

- **Features.** The encoder has no trained weight, so its output is reproducible across machines and ensembles; the 0–1 scaling uses the global extrema of the ensemble, so that two ensembles with different density ranges are not directly comparable in feature space unless scaled with the same `vmin`, `vmax` (`encode_ensemble` accepts them). The features are computed on `core`; set `padd = 0` to include the padding cells, as the 2b notebook did on the full Boulia grid.
- **Embedding.** UMAP with `n_neighbors = 5` preserves local structure: the strings seen in the embedding are chains of models that differ by one small change, not clusters in the usual sense, and DBSCAN with a small `EPS` cuts them where the chain is interrupted. A larger `n_neighbors` (15–30) gives blob-like clusters at the cost of the fine structure. The embedding is not deterministic across UMAP versions even with a fixed seed.
- **Clustering.** `EPS` must be chosen for each ensemble (the $k$-distance curve and the table of counts above); the same value gives different numbers of clusters for different $N$ and different UMAP parameters. HDBSCAN removes this choice but tends to merge the strings.
- **Reading the clusters.** With the synthetic ensemble, the enrichment table shows which bodies each cluster is made of. On a real ensemble, the motif enrichment of the motif workflow (`ma.cluster_enrichment(df, labels)`) plays the same role; the labels saved by this notebook (`<prefix>_cluster_labels.npy`) are the `labels_all` that the motif notebook expects, provided both run on the same `core` and the same model order.

### References

- Campello, R.J.G.B., Moulavi, D. & Sander, J., 2013. Density-based clustering based on hierarchical density estimates, *PAKDD*, LNCS 7819, 160–172.
- Ester, M., Kriegel, H.-P., Sander, J. & Xu, X., 1996. A density-based algorithm for discovering clusters in large spatial databases with noise, *Proc. KDD-96*, 226–231.
- Johnson, W.B. & Lindenstrauss, J., 1984. Extensions of Lipschitz mappings into a Hilbert space, *Contemp. Math.*, 26, 189–206.
- McInnes, L., Healy, J. & Melville, J., 2018. UMAP: Uniform Manifold Approximation and Projection for dimension reduction, arXiv:1802.03426.